# Build a spec one property at a time

Run one section at a time; stop to drag and vote.

Needs `suggest-search`, Node, and an authenticated `codex` CLI.

In [ ]:
from pathlib import Path
import sys

# Use this checkout from demos/, spytial-py/, or spytial-org/.
cwd = Path.cwd()
repo = next((p for p in (cwd, cwd.parent, cwd / "spytial-py")
             if (p / "spytial" / "suggest" / "_session.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the spytial-py checkout or its demos folder.")
sys.path.insert(0, str(repo))

from dataclasses import dataclass
from typing import Optional
from hypothesis import strategies as st
from IPython.display import display
import spytial
from spytial.suggest import suggest, SuggestSession, SpecUnavailableError

In [ ]:
@dataclass
class Tree:
    value: int
    left: Optional["Tree"] = None
    right: Optional["Tree"] = None

examples = [
    Tree(8, Tree(3), Tree(10)),
    Tree(8, Tree(3, Tree(1), Tree(6)), Tree(10, None, Tree(14))),
]

# Generate small trees without cycles or shared children.
def trees(depth):
    children = st.none() if depth == 0 else st.one_of(st.none(), trees(depth - 1))
    return st.builds(Tree, value=st.integers(0, 20), left=children, right=children)

strategy = trees(2)

## Start with data

No model request yet. You can enter properties in the view or use the cells below.

In [ ]:
session = suggest(Tree, examples=examples, strategy=strategy, max_examples=10)
spec = session.spec
display(session)

## First property

In [ ]:
vertical = session.add_property("Parents should be above their children")

Drag and vote above, then click **I'm done** or run this. It combines every supported survivor when the combination fits your judgments.

In [ ]:
checkpoint = vertical.done()
{
    "status": checkpoint["status"],
    "accepted interpretations": len(checkpoint["accepted_ids"]),
    "applied to class": spec.applied,
}

## Apply and draw

If unfinished, use **Resume** or **History**, then retry.

In [ ]:
try:
    spec = session.apply()
except SpecUnavailableError as error:
    print(error)
else:
    spytial.diagram(examples[1])

Inspect the current accepted rules (optional):

In [ ]:
try:
    print(spec)
except SpecUnavailableError as error:
    print(error)

## Add a property

Once the first property is accepted, it stays applied to both comparison diagrams.

In [ ]:
horizontal = session.add_property("Align siblings horizontally")

Vote above, then end this round:

In [ ]:
horizontal.done()["status"]

Only accepted properties enter the living spec. Draw the same tree with its current rules:

In [ ]:
try:
    spytial.diagram(examples[1], spec=spec)
except SpecUnavailableError as error:
    print(error)

## Revisit a property

This replaces its candidates and reuses its judgments. Other accepted properties stay active.

In [ ]:
vertical.revise("Keep left children below and left, and right children below and right")

Judge the revised property above, then:

In [ ]:
vertical.done()["status"]

In [ ]:
try:
    spytial.diagram(examples[1], spec=spec)
except SpecUnavailableError as error:
    print(error)

## Save and close the controls

**I'm done** pauses. `close()` stops the controls. A supported spec still draws.

In [ ]:
save_path = Path("tree-session.json")
session.save(save_path)
session.close()
print(f"Controls closed. Saved to {save_path.resolve()}")

In [ ]:
try:
    spytial.diagram(examples[1], spec=spec)
except SpecUnavailableError as error:
    print(error)

## Reopen

Votes and properties return. Rerun **Apply and draw** to bind the restored session.

In [ ]:
session = SuggestSession.load(Tree, save_path, strategy=strategy, max_examples=10)
spec = session.spec
vertical, horizontal = session.properties[:2]
session.resume()
display(session)